# Medication Prediction Analysis

This notebook implements machine learning models for predicting medication outcomes using:
- CatBoost gradient boosting
- Keras neural networks 
- Super Learner ensemble combining both models

The analysis includes internal validation on research cohorts and external validation across different datasets.

## Setup and Imports

In [1]:
# Standard library imports
import os
import pandas as pd
import numpy as np
from typing import List, Dict

# Custom data loading functions
from utils.load_data import (
    load_ppp, load_pond, load_hbn, load_abcd, root_dir
)

# Custom model training and evaluation modules
from models.catboost_trainer import CatBoostTrainer
from models.keras_trainer import KerasTrainer
from models.super_learner import SuperLearner

# Utility modules
from utils.experiment_manager import (
    generate_experiment_data, 
    save_command_scripts
)
from utils.evaluation import (
    calculate_summary_statistics,
    format_confidence_interval,
    calculate_super_learner_fairness,
    plot_roc_auc,
    calculate_super_learner_shap_values, plot_shap_beeswarm
)

## Configuration Parameters

In [2]:
# Experimental design parameters
N_SUBSAMPLES = 10  # Number of balanced subsamples per outcome
N_FOLDS = 5        # Number of cross-validation folds

# Universal control variables
RUN_HYPERPARAMETER_TUNING = True  # Set to False to skip tuning and use defaults

# Random seed for reproducibility
RANDOM_SEED = 42

# Output directory sufficx
OUTPUT_SUFFIX = '_' + str(N_SUBSAMPLES) + 'bootstraps'
pond_output_dir = os.path.join(root_dir() + OUTPUT_SUFFIX, 'POND')
hbn_output_dir = os.path.join(root_dir() + OUTPUT_SUFFIX, 'HBN')
abcd_output_dir = os.path.join(root_dir() + OUTPUT_SUFFIX, 'ABCD')
ppp_output_dir = os.path.join(root_dir() + OUTPUT_SUFFIX, 'PPP')

## Research Cohort Analysis

### POND Dataset

#### Load Data

In [3]:
# Load POND dataset
pond_df, pond_features, pond_outcomes = load_pond()

# Identify continuous features (those with > 5 unique values)
pond_continuous_features = [
    feature for feature in pond_features 
    if pond_df[feature].nunique() > 5
]

print(f"POND Dataset:")
print(f"- {len(pond_df)} total samples")
print(f"- {len(pond_features)} features ({len(pond_continuous_features)} continuous)")
print(f"- {len(pond_outcomes)} outcome variables: {pond_outcomes}")

POND Dataset:
- 598 total samples
- 136 features (13 continuous)
- 3 outcome variables: ['stimulant_outcome', 'antidepressant_outcome', 'antipsychotic_outcome']


#### Generate Hyperparameter Tuning Commands

In [5]:
# Generate balanced subsamples, CV splits, and tuning commands
tuning_commands = generate_experiment_data(
    df=pond_df,
    subject_col='subject',
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    base_dir=pond_output_dir
)

# Only generate and save tuning scripts if tuning is enabled
if RUN_HYPERPARAMETER_TUNING:
    save_command_scripts(tuning_commands, pond_output_dir)
    print("Run hyperparameter tuning using the generated shell scripts.")
    print("Set RUN_HYPERPARAMETER_TUNING = False to skip tuning and use default parameters.")
else:
    print("Hyperparameter tuning is disabled. Using default parameters:")
    print(f"- CatBoost: 100 iterations, 0.1 learning rate, depth 4")
    print(f"- Keras: 2 layers, 64 units, 0.001 learning rate")

Run: parallel --progress :::: /d/mjt/5/marlee/temp/data/revision_10bootstraps/POND/run_catboost_tuning.sh
Run: parallel --progress :::: /d/mjt/5/marlee/temp/data/revision_10bootstraps/POND/run_keras_tuning.sh
Run hyperparameter tuning using the generated shell scripts.
Set RUN_HYPERPARAMETER_TUNING = False to skip tuning and use default parameters.


#### Model Training and Internal Validation: Base Models

In [4]:
# Initialize model trainers
catboost_trainer = CatBoostTrainer(pond_output_dir)
keras_trainer = KerasTrainer(pond_output_dir)
super_learner = SuperLearner(pond_output_dir)

print("Training and evaluating models on POND dataset...")

# Train base learners (needed for Super Learner)
pond_catboost_results, pond_catboost_tprs = catboost_trainer.train_and_evaluate_internal(
    df=pond_df,
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    use_tuned_params=RUN_HYPERPARAMETER_TUNING
)

pond_keras_results, pond_keras_tprs = keras_trainer.train_and_evaluate_internal(
    df=pond_df,
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    use_tuned_params=RUN_HYPERPARAMETER_TUNING
)

print("POND internal validation completed for base learners.")

Training and evaluating models on POND dataset...
Input validation passed: 598 samples, 136 features, 3 outcomes
Input validation passed: 598 samples, 136 features, 3 outcomes
POND internal validation completed for base learners.


#### Model Training and Internal Validation: Base Models

In [5]:
# Train Super Learner
pond_super_learner_results, pond_super_learner_tprs = super_learner.train_and_evaluate_internal(
    df=pond_df,
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS
)

print("POND internal validation completed for super learners.")

Input validation passed: 598 samples, 136 features, 3 outcomes
POND internal validation completed for super learners.


#### Results - Super Learner

In [6]:
# Calculate summary statistics
metrics_to_analyze = ['ROC AUC', 'PR AUC', 'Accuracy', 'Sensitivity', 'Specificity']

print("=== POND Internal Validation Results - Super Learner ===")

summary = pond_super_learner_results.groupby('Outcome')[metrics_to_analyze].apply(
    lambda x: calculate_summary_statistics(x, metrics_to_analyze)
)

for outcome in pond_outcomes:
    if outcome in summary.index:
        row = summary.loc[outcome]
        formatted_auc = format_confidence_interval(
            row['ROC AUC_median'],
            row['ROC AUC_q1'],
            row['ROC AUC_q3']
        )
        print(f"\n{outcome}:")
        print(f"  ROC AUC: {formatted_auc}")

=== POND Internal Validation Results - Super Learner ===

stimulant_outcome:
  ROC AUC: 0.754 (0.729-0.798)

antidepressant_outcome:
  ROC AUC: 0.828 (0.782-0.869)

antipsychotic_outcome:
  ROC AUC: 0.790 (0.720-0.856)


### External Validation - HBN Dataset

#### Load Data

In [7]:
# Load HBN dataset for external validation
hbn_df = load_hbn()

print(f"HBN Dataset: {len(hbn_df)} samples for external validation")

# Evaluate POND-trained models on HBN dataset
print("Evaluating POND models on HBN dataset...")

# Evaluate Super Learner
hbn_super_learner_results, hbn_super_learner_tprs = super_learner.evaluate_external(
    external_df=hbn_df,
    subject_col='EID',
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    trained_model_dir=pond_output_dir,
    output_dir=hbn_output_dir
)

print("HBN external validation completed.")

HBN Dataset: 1764 samples for external validation
Evaluating POND models on HBN dataset...
Input validation passed: 1764 samples, 136 features, 3 outcomes
HBN external validation completed.


#### Results - Super Learner

In [8]:
print("=== HBN External Validation Results - Super Learner ===")

summary = hbn_super_learner_results.groupby('Outcome')[metrics_to_analyze].apply(
    lambda x: calculate_summary_statistics(x, metrics_to_analyze)
)

for outcome in pond_outcomes:
    if outcome in summary.index:
        row = summary.loc[outcome]
        formatted_auc = format_confidence_interval(
            row['ROC AUC_median'],
            row['ROC AUC_q1'],
            row['ROC AUC_q3']
        )
        print(f"\n{outcome}:")
        print(f"  ROC AUC: {formatted_auc}")

=== HBN External Validation Results - Super Learner ===

stimulant_outcome:
  ROC AUC: 0.696 (0.674-0.711)

antidepressant_outcome:
  ROC AUC: 0.753 (0.737-0.774)

antipsychotic_outcome:
  ROC AUC: 0.794 (0.754-0.812)


### External Validation - ABCD Dataset

#### Load Data

In [9]:
# Load ABCD dataset for external validation
abcd_df = load_abcd()

print(f"ABCD Dataset: {len(abcd_df)} samples for external validation")

# Evaluate POND-trained models on ABCD dataset
print("Evaluating POND models on ABCD dataset...")

# Evaluate Super Learner
abcd_super_learner_results, abcd_super_learner_tprs = super_learner.evaluate_external(
    external_df=abcd_df,
    subject_col='src_subject_id',
    feature_cols=pond_features,
    continuous_features=pond_continuous_features,
    outcome_vars=pond_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    trained_model_dir=pond_output_dir,
    output_dir=abcd_output_dir
)

print("ABCD external validation completed.")

ABCD Dataset: 2396 samples for external validation
Evaluating POND models on ABCD dataset...
Input validation passed: 2396 samples, 136 features, 3 outcomes
ABCD external validation completed.


#### Results - Super Learner

In [10]:
print("=== ABCD External Validation Results - Super Learner ===")

summary = abcd_super_learner_results.groupby('Outcome')[metrics_to_analyze].apply(
    lambda x: calculate_summary_statistics(x, metrics_to_analyze)
)

for outcome in pond_outcomes:
    if outcome in summary.index:
        row = summary.loc[outcome]
        formatted_auc = format_confidence_interval(
            row['ROC AUC_median'],
            row['ROC AUC_q1'],
            row['ROC AUC_q3']
        )
        print(f"\n{outcome}:")
        print(f"  ROC AUC: {formatted_auc}")

=== ABCD External Validation Results - Super Learner ===

stimulant_outcome:
  ROC AUC: 0.702 (0.696-0.713)

antidepressant_outcome:
  ROC AUC: 0.673 (0.650-0.694)

antipsychotic_outcome:
  ROC AUC: 0.809 (0.787-0.832)


#### Fairness

In [11]:
print("Calculating Super Learner fairness metrics...")

# Binary sensitive attributes
sensitive_cols = ['sex', 'intellectual_disability', 'White_Minority', 'income']

# === Cross-cohort analysis (POND + HBN + ABCD) ===
cross_cohort_data = [
    {
        'name': 'POND',
        'df': pond_df,
        'feature_cols': pond_features,
        'continuous_features': pond_continuous_features,
        'model_dir': pond_output_dir,
        'n_subsamples': N_SUBSAMPLES,
        'n_folds': N_FOLDS
    },
    {
        'name': 'HBN',
        'df': hbn_df,
        'feature_cols': pond_features,  # Using POND features for consistency
        'continuous_features': pond_continuous_features,
        'model_dir': pond_output_dir,  # Using POND-trained models
        'n_subsamples': N_SUBSAMPLES,
        'n_folds': N_FOLDS
    },
    {
        'name': 'ABCD',
        'df': abcd_df,
        'feature_cols': pond_features,  # Using POND features for consistency
        'continuous_features': pond_continuous_features,
        'model_dir': pond_output_dir,  # Using POND-trained models
        'n_subsamples': N_SUBSAMPLES,
        'n_folds': N_FOLDS
    }
]

cross_cohort_fairness = calculate_super_learner_fairness(
    cohort_data_list=cross_cohort_data,
    outcome_vars=pond_outcomes,
    sensitive_cols=sensitive_cols
)

print("Cross-Cohort Fairness Results:")
print(cross_cohort_fairness.round(3))

Calculating Super Learner fairness metrics...
Cross-Cohort Fairness Results:
                   Outcome      Sensitive_Attribute    DPR    EOR  N_total  \
0        stimulant_outcome                      sex  0.702  0.696    10260   
1        stimulant_outcome  intellectual_disability  0.719  0.627    10260   
2        stimulant_outcome           White_Minority  0.948  0.943     9622   
3        stimulant_outcome                   income  0.937  0.845     7782   
4   antidepressant_outcome                      sex  0.690  0.679     5760   
5   antidepressant_outcome  intellectual_disability  0.708  0.647     5760   
6   antidepressant_outcome           White_Minority  0.566  0.708     5403   
7   antidepressant_outcome                   income  0.718  0.767     4422   
8    antipsychotic_outcome                      sex  0.807  0.827     3120   
9    antipsychotic_outcome  intellectual_disability  0.571  0.744     3120   
10   antipsychotic_outcome           White_Minority  0.831  0.895

#### AU-ROC Figure

In [24]:
# POND
plot_roc_auc(pond_output_dir, pond_super_learner_tprs)

# HBN
plot_roc_auc(hbn_output_dir, hbn_super_learner_tprs)

# ABCD
plot_roc_auc(abcd_output_dir, abcd_super_learner_tprs)

## Electronic Medical Records Analysis - PPP Dataset

#### Load Data

In [27]:
# Load PPP (EMR) dataset
ppp_df, ppp_features, ppp_outcomes, ppp_df_without_dummies = load_ppp()

# For EMR data, typically only age is continuous
ppp_continuous_features = ['age']

print(f"PPP EMR Dataset:")
print(f"- {len(ppp_df)} total samples")
print(f"- {len(ppp_features)} features ({len(ppp_continuous_features)} continuous)")
print(f"- {len(ppp_outcomes)} outcome variables: {ppp_outcomes}")

# Proportion of unknown
prop_unknown = (ppp_df_without_dummies == 'unknown').sum()/len(ppp_df_without_dummies)
prop_rows_unknown = (ppp_df_without_dummies == 'unknown').any(axis=1).mean()
print(f" - Max % unknown in a feature: {np.max(prop_unknown*100)}")
print(f" - % participants with at least one unknown: {prop_rows_unknown*100}")

PPP EMR Dataset:
- 312 total samples
- 104 features (1 continuous)
- 3 outcome variables: ['stimulant_outcome', 'antidepressant_outcome', 'antipsychotic_outcome']
 - Max % unknown in a feature: 28.846153846153843
 - % participants with at least one unknown: 64.74358974358975


### Generate Hyperparameter Tuning Commands

In [15]:
# Generate balanced subsamples, CV splits, and tuning commands for PPP
ppp_tuning_commands = generate_experiment_data(
    df=ppp_df,
    subject_col='study_id2',
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=ppp_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    base_dir=ppp_output_dir
)

# Only generate and save tuning scripts if tuning is enabled
if RUN_HYPERPARAMETER_TUNING:
    save_command_scripts(ppp_tuning_commands, ppp_output_dir)
    print("Run hyperparameter tuning using the generated shell scripts.")
    print("Set RUN_HYPERPARAMETER_TUNING = False to skip tuning and use default parameters.")
else:
    print("Hyperparameter tuning is disabled. Using default parameters:")
    print(f"- CatBoost: 100 iterations, 0.1 learning rate, depth 4")
    print(f"- Keras: 2 layers, 64 units, 0.001 learning rate")

Run: parallel --progress :::: /d/mjt/5/marlee/temp/data/revision_10bootstraps/PPP/run_catboost_tuning.sh
Run: parallel --progress :::: /d/mjt/5/marlee/temp/data/revision_10bootstraps/PPP/run_keras_tuning.sh
Run hyperparameter tuning using the generated shell scripts.
Set RUN_HYPERPARAMETER_TUNING = False to skip tuning and use default parameters.


### Model Training and Internal Validation: Base Learners

In [28]:
# Initialize model trainers for PPP dataset
ppp_catboost_trainer = CatBoostTrainer(ppp_output_dir)
ppp_keras_trainer = KerasTrainer(ppp_output_dir)
ppp_super_learner = SuperLearner(ppp_output_dir)

print("Training and evaluating models on PPP EMR dataset...")

# Train base learners (needed for Super Learner)
ppp_catboost_results, ppp_catboost_tprs = ppp_catboost_trainer.train_and_evaluate_internal(
    df=ppp_df,
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=ppp_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    use_tuned_params=RUN_HYPERPARAMETER_TUNING
)

ppp_keras_results, ppp_keras_tprs = ppp_keras_trainer.train_and_evaluate_internal(
    df=ppp_df,
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=ppp_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS,
    use_tuned_params=RUN_HYPERPARAMETER_TUNING
)

print("PPP EMR internal validation completed for base learners.")

Training and evaluating models on PPP EMR dataset...
Input validation passed: 312 samples, 104 features, 3 outcomes
Input validation passed: 312 samples, 104 features, 3 outcomes
PPP EMR internal validation completed for base learners.


### Model Training and Internal Validation: Meta Learner

In [29]:
# Train Super Learner
ppp_super_learner_results, ppp_super_learner_tprs = ppp_super_learner.train_and_evaluate_internal(
    df=ppp_df,
    feature_cols=ppp_features,
    continuous_features=ppp_continuous_features,
    outcome_vars=ppp_outcomes,
    n_subsamples=N_SUBSAMPLES,
    n_folds=N_FOLDS
)

print("PPP EMR internal validation completed for meta learner.")

Input validation passed: 312 samples, 104 features, 3 outcomes
PPP EMR internal validation completed for meta learner.


### Results - Super Learner

In [30]:
print("=== PPP EMR Internal Validation Results - Super Learner ===")

summary = ppp_super_learner_results.groupby('Outcome')[metrics_to_analyze].apply(
    lambda x: calculate_summary_statistics(x, metrics_to_analyze)
)

for outcome in ppp_outcomes:
    if outcome in summary.index:
        row = summary.loc[outcome]
        formatted_auc = format_confidence_interval(
            row['ROC AUC_median'],
            row['ROC AUC_q1'],
            row['ROC AUC_q3']
        )
        print(f"\n{outcome}:")
        print(f"  ROC AUC: {formatted_auc}")

=== PPP EMR Internal Validation Results - Super Learner ===

stimulant_outcome:
  ROC AUC: 0.842 (0.809-0.881)

antidepressant_outcome:
  ROC AUC: 0.824 (0.771-0.874)

antipsychotic_outcome:
  ROC AUC: 0.873 (0.829-0.905)


### Fairness

In [31]:
print("Calculating Super Learner fairness metrics...")

# Binary sensitive attributes
sensitive_cols = ['sex', 'intellectual_disability']
ppp_data = [
    {
        'name': 'PPP',
        'df': ppp_df,
        'feature_cols': ppp_features,
        'continuous_features': ppp_continuous_features,
        'model_dir': ppp_output_dir,
        'n_subsamples': N_SUBSAMPLES,
        'n_folds': N_FOLDS
    }
]

ppp_fairness = calculate_super_learner_fairness(
    cohort_data_list=ppp_data,
    outcome_vars=ppp_outcomes,
    sensitive_cols=sensitive_cols
)

print("PPP Fairness Results:")
print(ppp_fairness.round(3))

Calculating Super Learner fairness metrics...
PPP Fairness Results:
                  Outcome      Sensitive_Attribute    DPR    EOR  N_total  \
0       stimulant_outcome                      sex  0.790  0.489     2280   
1       stimulant_outcome  intellectual_disability  0.452  0.285     2280   
2  antidepressant_outcome                      sex  0.791  0.723     1640   
3  antidepressant_outcome  intellectual_disability  0.620  0.409     1640   
4   antipsychotic_outcome                      sex  0.911  0.811     2460   
5   antipsychotic_outcome  intellectual_disability  0.616  0.586     2460   

   N_group_0  N_group_1  N_group_2  Pct_pred_pos_group_0  \
0        345       1935          0                36.232   
1       1046        517        717                61.090   
2        281       1359          0                55.160   
3        772        345        523                53.756   
4        389       2071          0                53.985   
5       1022        561        8

#### AU-ROC Figure

In [36]:
# PPP
plot_roc_auc(ppp_output_dir, ppp_super_learner_tprs)

### SHAP

In [44]:
# Process each outcome
for outcome in ppp_outcomes:
    print(f"\nProcessing {outcome}")
    
    try:
        # Calculate SHAP values using correct pipeline
        shap_df = calculate_super_learner_shap_values(
            df=ppp_df,
            feature_cols=ppp_features,
            continuous_features=ppp_continuous_features,
            outcome_var=outcome,
            model_dir=ppp_output_dir,
            n_subsamples=N_SUBSAMPLES,
            n_folds=N_FOLDS
        )
        
        print(f"SHAP DataFrame shape: {shap_df.shape}")
        print(f"Samples with SHAP values: {(shap_df['count'] > 0).sum()}")
        
        # Create beeswarm plot
        plot_shap_beeswarm(
            shap_df=shap_df,
            df=ppp_df,
            feature_cols=ppp_features,
            outcome_var=outcome,
            save_dir=ppp_output_dir,
            top_n=10
        )
        
        # Save SHAP DataFrame
        shap_df.to_csv(os.path.join(ppp_output_dir, f'{outcome}_shap_values.csv'), index=False)
        
        print(f"Completed {outcome}")
        
    except Exception as e:
        print(f"Error processing {outcome}: {e}")
        continue

print(f"\nSHAP analysis complete!")
print(f"Results saved to: {ppp_output_dir}")


Processing stimulant_outcome
Computing Super Learner SHAP values for stimulant_outcome
Processing subsample 1/10
  Processing fold 1/5
  Processing fold 2/5
  Processing fold 3/5
  Processing fold 4/5
  Processing fold 5/5
Processing subsample 2/10
  Processing fold 1/5
  Processing fold 2/5
  Processing fold 3/5
  Processing fold 4/5
  Processing fold 5/5
Processing subsample 3/10
  Processing fold 1/5
  Processing fold 2/5
  Processing fold 3/5
  Processing fold 4/5
  Processing fold 5/5
Processing subsample 4/10
  Processing fold 1/5
  Processing fold 2/5
  Processing fold 3/5
  Processing fold 4/5
  Processing fold 5/5
Processing subsample 5/10
  Processing fold 1/5
  Processing fold 2/5
  Processing fold 3/5
  Processing fold 4/5
  Processing fold 5/5
Processing subsample 6/10
  Processing fold 1/5
  Processing fold 2/5
  Processing fold 3/5
  Processing fold 4/5
  Processing fold 5/5
Processing subsample 7/10
  Processing fold 1/5
  Processing fold 2/5
  Processing fold 3/5
  Pr

## Results Summary

In [33]:
print("\n" + "="*60)
print("COMPREHENSIVE RESULTS SUMMARY - SUPER LEARNER ONLY")
print("="*60)

datasets_and_results = [
    ("POND Research Cohort (Internal)", pond_super_learner_results, pond_outcomes),
    ("HBN Research Cohort (External)", hbn_super_learner_results, pond_outcomes),
    ("ABCD Research Cohort (External)", abcd_super_learner_results, pond_outcomes),
    ("PPP EMR Cohort (Internal)", ppp_super_learner_results, ppp_outcomes)
]

for dataset_name, results, outcomes in datasets_and_results:
    print(f"\n{dataset_name}:")
    
    if len(results) > 0:
        summary = results.groupby('Outcome')[metrics_to_analyze].apply(
            lambda x: calculate_summary_statistics(x, metrics_to_analyze)
        )
        
        for outcome in outcomes:
            if outcome in summary.index:
                row = summary.loc[outcome]
                formatted_auc = format_confidence_interval(
                    row['ROC AUC_median'],
                    row['ROC AUC_q1'],
                    row['ROC AUC_q3']
                )
                print(f"  {outcome}: {formatted_auc}")
    else:
        print("  No results available")


COMPREHENSIVE RESULTS SUMMARY - SUPER LEARNER ONLY

POND Research Cohort (Internal):
  stimulant_outcome: 0.754 (0.729-0.798)
  antidepressant_outcome: 0.828 (0.782-0.869)
  antipsychotic_outcome: 0.790 (0.720-0.856)

HBN Research Cohort (External):
  stimulant_outcome: 0.696 (0.674-0.711)
  antidepressant_outcome: 0.753 (0.737-0.774)
  antipsychotic_outcome: 0.794 (0.754-0.812)

ABCD Research Cohort (External):
  stimulant_outcome: 0.702 (0.696-0.713)
  antidepressant_outcome: 0.673 (0.650-0.694)
  antipsychotic_outcome: 0.809 (0.787-0.832)

PPP EMR Cohort (Internal):
  stimulant_outcome: 0.842 (0.809-0.881)
  antidepressant_outcome: 0.824 (0.771-0.874)
  antipsychotic_outcome: 0.873 (0.829-0.905)
